In [1]:
import torch
import numpy as np
from datasets import Dataset
from transformers import pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm
import os

In [2]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)


device: mps


In [ ]:
from tablevault import tablevault

vault = tablevault.Vault(user_id="jinjin",
                            process_name="bidirectional_averaged_zero_shot_mrpc",
                            arango_url="http://localhost:8629",
                            arango_db="tv_experiment_1",
                            arango_username="tablevault_user",
                            arango_password="tablevault_password",
                            new_arango_db=False,               
                            arango_root_username="root",
                            arango_root_password="passwd",
                            description_embedding_size=3072,
                        )

from openai import OpenAI


openai_key_file = "/Users/jinjinzhao/Documents/work_projects/my_keys/my_keys/openai_jinjin.key"
with open(openai_key_file, 'r') as f:
    openai_key = f.read()

os.environ["OPENAI_API_KEY"] = openai_key

client = OpenAI()

In [ ]:
def get_embeddings(text):
    return client.embeddings.create(
            input=text,
            model="text-embedding-3-large"
        ).data[0].embedding

In [3]:
model_name = "typeform/distilbert-base-uncased-mnli"

zsc = pipeline(
    "zero-shot-classification",
    model=model_name,
    tokenizer=model_name,
    framework="pt",
    device=device,
)

print(model_name)
print(zsc.model.config.id2label)


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

typeform/distilbert-base-uncased-mnli
{0: 'ENTAILMENT', 1: 'NEUTRAL', 2: 'CONTRADICTION'}


In [4]:
ds = vault.query_item_content("glue_mrpc_validation")
ds = Dataset.from_dict(ds)
print(ds)
print(ds[0])

sent1 = ds["sentence1"]
sent2 = ds["sentence2"]
y_true = np.array(ds["label"])

forward_texts = [f"Sentence 1: {s1}\nSentence 2: {s2}" for s1, s2 in zip(sent1, sent2)]
reverse_texts = [f"Sentence 1: {s2}\nSentence 2: {s1}" for s1, s2 in zip(sent1, sent2)]

print("num_examples:", len(y_true))
print("positive_rate:", y_true.mean())
print(forward_texts[0])
print(reverse_texts[0])


Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 408
})
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}
num_examples: 408
positive_rate: 0.6838235294117647
Sentence 1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
Sentence 2: " The foodservice pie business does not fit our long-term growth strategy .
Sentence 1: " The foodservice pie business does not fit our long-term growth strategy .
Sentence 2: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .


In [5]:
candidate_labels = ["paraphrase", "not paraphrase"]
hypothesis_template = "These two sentences are {}."
batch_size = 32

def get_paraphrase_scores(texts, batch_size=32):
    scores = []
    for start in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[start:start + batch_size]
        outputs = zsc(
            batch_texts,
            candidate_labels=candidate_labels,
            hypothesis_template=hypothesis_template,
            multi_label=False,
            batch_size=batch_size,
            truncation=True,
        )
        if isinstance(outputs, dict):
            outputs = [outputs]
        for out in outputs:
            label_to_score = dict(zip(out["labels"], out["scores"]))
            scores.append(float(label_to_score["paraphrase"]))
    return np.array(scores)

forward_scores = get_paraphrase_scores(forward_texts, batch_size=batch_size)
reverse_scores = get_paraphrase_scores(reverse_texts, batch_size=batch_size)
avg_scores = (forward_scores + reverse_scores) / 2.0
y_pred = (avg_scores >= 0.5).astype(int)

print("done")


  0%|          | 0/13 [00:00<?, ?it/s]

  0%|          | 0/13 [00:00<?, ?it/s]

done


In [ ]:
vault.create_record_list("distilbert_bidirectional_prediction", column_names=["prediction", "forward_scores" , "reverse_scores"])

for i in range(len(y_pred)):
    vault.append_record("distilbert_bidirectional_prediction", 
                        {
                            "prediction": y_pred[i],
                            "forward_scores": float(forward_scores[i]),
                            "reverse_scores": float(reverse_scores[i]),
                        },
                       input_items = {
                           "glue_mrpc_validation": [i, i + 1],
                       }
                       )

description = "Per-example prediction dataset for the GLUE MRPC validation split produced by a bidirectional zero-shot paraphrase workflow using typeform/distilbert-base-uncased-mnli. Each record corresponds to one sentence pair from glue_mrpc_validation and stores: prediction (final binary paraphrase label, obtained by averaging the forward and reverse paraphrase scores and thresholding at 0.5), forward_scores (paraphrase score for the original sentence order), and reverse_scores (paraphrase score for the swapped sentence order). This dataset is the main model output table used to analyze individual predictions and to compute aggregate evaluation metrics summarized in bidirectional_averaged_zero_shot_mrpc_summary."
embedding = get_embeddings(description)
vault.create_description("distilbert_bidirectional_prediction", description, embedding)

properties = {"task": "paraphrase detection", "dataset_type": "model predictions", "source": "glue/mrpc", "split": "validation", "size": "408", "model": "typeform/distilbert-base-uncased-mnli", "method": "bidirectional averaged zero-shot classification", "input_format": "sentence pair", "labels": "binary", "output_columns": "prediction, forward_scores, reverse_scores"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("distilbert_bidirectional_prediction", cat, embedding, prop)

In [6]:
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
report = classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"])
print({"accuracy": acc, "f1": f1})
print(classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"]))


{'accuracy': 0.6127450980392157, 'f1': 0.7366666666666667}
                precision    recall  f1-score   support

not_paraphrase       0.33      0.22      0.27       129
    paraphrase       0.69      0.79      0.74       279

      accuracy                           0.61       408
     macro avg       0.51      0.51      0.50       408
  weighted avg       0.58      0.61      0.59       408



In [7]:
for i in range(5):
    print("=" * 80)
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("forward_score:", float(forward_scores[i]))
    print("reverse_score:", float(reverse_scores[i]))
    print("avg_score:", float(avg_scores[i]))
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))


sentence1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
sentence2: " The foodservice pie business does not fit our long-term growth strategy .
forward_score: 0.533607542514801
reverse_score: 0.45817941427230835
avg_score: 0.4958934783935547
true: 1 pred: 0
sentence1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war .
sentence2: His wife said he was " 100 percent behind George Bush " and looked forward to using his years of training in the war .
forward_score: 0.5474331974983215
reverse_score: 0.6203325390815735
avg_score: 0.5838828682899475
true: 0 pred: 1
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 yen JPY = , virtually flat on the session , and at 1.2871 against the Swiss franc CHF = , down 0.1 percent .
forward_score: 0.5974612236022949
reverse_score:

In [8]:
mistakes = np.where(y_true != y_pred)[0][:10]
print("num_errors:", int((y_true != y_pred).sum()))

for i in mistakes:
    print("=" * 80)
    print("idx:", int(i))
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("forward_score:", float(forward_scores[i]))
    print("reverse_score:", float(reverse_scores[i]))
    print("avg_score:", float(avg_scores[i]))
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))


num_errors: 158
idx: 0
sentence1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
sentence2: " The foodservice pie business does not fit our long-term growth strategy .
forward_score: 0.533607542514801
reverse_score: 0.45817941427230835
avg_score: 0.4958934783935547
true: 1 pred: 0
idx: 1
sentence1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war .
sentence2: His wife said he was " 100 percent behind George Bush " and looked forward to using his years of training in the war .
forward_score: 0.5474331974983215
reverse_score: 0.6203325390815735
avg_score: 0.5838828682899475
true: 0 pred: 1
idx: 2
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 yen JPY = , virtually flat on the session , and at 1.2871 against the Swiss franc CHF = , down 0.1 percent .
forward_sco

In [9]:
vault.create_record_list("bidirectional_averaged_zero_shot_mrpc_summary", column_names=["accuracy", "f1", "classification_report"])


summary = {
    "accuracy": float(acc),
    "f1": float(f1),
    "classification_report": str(report)
}

vault.append_record("bidirectional_averaged_zero_shot_mrpc_summary", summary,
                    input_items = {
                        "glue_mrpc_validation": [0, len(ds)],
                        "distilbert_bidirectional_prediction": [0, len(ds)]
                    })

summary

description = "Summary dataset for the bidirectional averaged zero-shot MRPC evaluation. It contains experiment-level performance results computed on the GLUE MRPC validation set after scoring each sentence pair in both forward and reversed order with the zero-shot model typeform/distilbert-base-uncased-mnli, averaging the two paraphrase scores, and converting the average score to a binary prediction using a 0.5 threshold.\n\nStructure: one record with three fields: accuracy (float), f1 (float), and classification_report (string containing the full per-class sklearn classification report).\n\nRole in the workflow: this dataset serves as the final evaluation summary for the prediction dataset distilbert_bidirectional_prediction, providing a compact record of overall model performance derived from the true MRPC labels and the generated bidirectional paraphrase predictions."
embedding = get_embeddings(description)
vault.create_description("bidirectional_averaged_zero_shot_mrpc_summary", description, embedding)

properties = {"task": "paraphrase detection", "dataset_role": "evaluation summary", "source": "glue/mrpc", "split": "validation", "size": "408", "model": "typeform/distilbert-base-uncased-mnli", "method": "bidirectional averaged zero-shot classification", "input_format": "sentence pair", "candidate_labels": "paraphrase, not paraphrase", "hypothesis_template": "These two sentences are {}.", "prediction_threshold": "0.5", "metrics": "accuracy, f1, classification_report", "upstream_dataset": "glue_mrpc_validation", "upstream_predictions": "distilbert_bidirectional_prediction"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("bidirectional_averaged_zero_shot_mrpc_summary", cat, embedding, prop)


{'dataset': 'glue/mrpc',
 'split': 'validation',
 'model': 'typeform/distilbert-base-uncased-mnli',
 'device': 'mps',
 'num_examples': 408,
 'accuracy': 0.6127450980392157,
 'f1': 0.7366666666666667}

In [ ]:
description = "This notebook runs a zero-shot paraphrase detection experiment on the GLUE MRPC validation set using the typeform/distilbert-base-uncased-mnli model through the Hugging Face zero-shot-classification pipeline. Its purpose is to estimate whether each sentence pair is a paraphrase without task-specific fine-tuning, while storing predictions, metadata, and summary results in TableVault. The workflow loads MRPC validation examples from TableVault, builds both forward and reversed sentence-pair prompts, scores each pair against the candidate labels paraphrase and not paraphrase with the hypothesis template \u201cThese two sentences are {}.\u201d, averages the forward and reverse paraphrase scores to reduce order sensitivity, and converts the averaged score to a binary prediction with a 0.5 threshold. It then saves per-example predictions and scores, computes evaluation metrics including accuracy, F1, and a classification report, inspects sample predictions and errors, and writes dataset-level summaries and process descriptions back to TableVault with OpenAI embedding-based metadata." # description of whole notebook
embedding = get_embeddings(description)
vault.create_description("bidirectional_averaged_zero_shot_mrpc", description, embedding)

properties = {"task": "paraphrase detection", "approach": "bidirectional averaged zero-shot classification", "model": "typeform/distilbert-base-uncased-mnli", "model_family": "DistilBERT", "inference_type": "zero-shot NLI", "dataset": "glue/mrpc", "dataset_split": "validation", "input_format": "sentence pair", "label_space": "paraphrase vs not paraphrase", "scoring": "average of forward and reverse paraphrase scores", "threshold": "0.5", "metrics": "accuracy, f1-score, classification report", "framework": "transformers pipeline, PyTorch", "device": "MPS or CPU", "tracking": "TableVault", "embedding_model": "text-embedding-3-large"} #e.g. model: distilbert-base-uncased-MRPC

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("bidirectional_averaged_zero_shot_mrpc", cat, embedding, prop)